In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex
from sentence_transformers import SentenceTransformer
import citation_utils
import metric_utils
import reranker_utils
import rrf
import hits_utils


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefin

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_002.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}


print("data loaded.")

data loaded.


In [3]:
# Qwen-embedding

# from FlagEmbedding import FlagReranker

# model = SentenceTransformer("/root/.cache/modelscope/hub/models/Qwen/Qwen3-Embedding-0___6B", model_kwargs={"torch_dtype": "float16"})

# dense_index = DenseIndex(model, "../data/processed/_dense_court", court_doc)

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation


In [4]:
# BGE-embedding

from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
sparse_index.load()

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

DenseIndex.embeddings:  (2107648, 1024)


In [9]:
from tqdm.notebook import tqdm
import reranker_utils
import hits_utils

RECALL_COUNT=1000
RERANK_COUNT=1000
NN = 10

test_001_df = pd.read_csv("../data/test_rewrite_001.csv")
test_001_d = {}
for query_id, query in zip(test_001_df['query_id'].tolist(), test_001_df['query'].tolist()):
    test_001_d[query_id] = query

id_l = []
citation_l = []
for query_id, query_l in tqdm(test_dict.items(), total=len(test_dict)):
    ranked_l_l = []
    # for query in query_l: 
    #     court_sparse_search_l = sparse_index.search_with_score(query, RECALL_COUNT)
    #     ranked_l_l.append([hit['citation'] for hit, score in court_sparse_search_l])
        # court_doc_score_l = hits_utils.merge_hits_with_score_l_by_max(court_dense_search_l, court_sparse_search_l)

    query_001 = test_001_d[query_id]
    court_dense_search_l = dense_index.search_with_score(query_001, RECALL_COUNT)
    ranked_l_l.append([hit['citation'] for hit, score in court_dense_search_l])
    court_sparse_search_l = sparse_index.search_with_score(query_001, RECALL_COUNT)
    ranked_l_l.append([hit['citation'] for hit, score in court_sparse_search_l])
                   
    rrf_result = rrf.compute2_with_score(ranked_l_l, k=60, top_k=10000)
    print('rrf_result.len:', len(rrf_result))
    
    court_doc_score_l = []
    for citation, score in rrf_result:
        if citation in court_consideration_d:
            court_doc_score_l.append(({'citation':citation, 'text':court_consideration_d[citation]}, score))

    hit_with_score_l = reranker_utils.rerank_by_dense_batch_chunked(reranker, query, [doc for doc,score in court_doc_score_l], 1000, 20, 384, 128)

    # rrf_score_l = rrf.compute2_with_score([[hit['citation'] for hit,_ in hit_with_score_l]], k=60, top_k=1000)

    first_layer = hit_with_score_l

    second_layer = citation_utils.compute_citation_score_with_sentence_pos(first_layer, decay="reciprocal")

    hits = []
    for citation, score in second_layer:
        if citation.lower().startswith("art."):
            hits.append({'citation':citation})

    print("first_layer.len:", len(first_layer),"second_layer.len:", len(second_layer), "hits.len:", len(hits))

    citations = [r['citation'] for r in hits] # hits is sorted 
    id_l.append(query_id)
    citation_l.append(';'.join(citations))
    print(query_id, len(hits))

  0%|          | 0/40 [00:00<?, ?it/s]

rrf_result.len: 1914
first_layer.len: 1000 second_layer.len: 467 hits.len: 208
test_001 208
rrf_result.len: 1869
first_layer.len: 1000 second_layer.len: 545 hits.len: 219
test_002 219
rrf_result.len: 1724
first_layer.len: 1000 second_layer.len: 1101 hits.len: 369
test_003 369
rrf_result.len: 1759
first_layer.len: 1000 second_layer.len: 804 hits.len: 301
test_004 301
rrf_result.len: 1756
first_layer.len: 1000 second_layer.len: 1547 hits.len: 521
test_005 521
rrf_result.len: 1939


KeyboardInterrupt: 

In [ ]:
import math
limit=40
query_id_l = []
predicted_citations_l = []
for query_id, predicted_citations in zip(id_l, citation_l):
    query_id_l.append(query_id)
    __l = predicted_citations.split(';')
    __limit = limit
    print(__limit, len(__l))
    predicted_citations_l.append(';'.join(__l[:__limit]))
sub_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':predicted_citations_l})
sub_df.to_csv("../output/submission.csv", index=False)